# 05 — Deep Blue data: Monte Carlo dose in the patient CT (GATE)

Notebook 04 computed the absorbed dose with a voxel S-value (VSV) kernel computed in water. Here the same TIA maps are used as voxelised sources of Lu-177 decays in a full Monte Carlo simulation (GATE 10, Geant4) in the patient anatomy, which is the reference for the dose engine.

For each patient two TIA maps of notebook 04 are simulated, **hybrid** and **MIM**, and the hybrid map is also simulated in a phantom of uniform water, so that the two effects that separate the VSV result from the patient Monte Carlo can be isolated:

| Comparison | What differs | Effect measured |
|---|---|---|
| MC water vs VSV | transport in water vs kernel convolution | accuracy of the convolution (kernel truncation at 40 mm, voxel sampling) |
| MC CT vs MC water | patient materials and densities vs water | heterogeneity / density effect |
| VSV/ρ vs MC CT | first-order density correction of the VSV dose | whether a density scaling is enough |
| hybrid vs MIM, MC and VSV | TIA map | whether the map differences of notebook 04 depend on the engine |

**Steps**
1. Inputs on the 4 mm grid of notebook 04: CT (HU) and TIA maps.
2. Monte Carlo runs with `lu177_patient_mc.py` (one simulation per configuration, cached on disk).
3. Scaling of the Monte Carlo dose to the TIA and statistical uncertainty.
4. Density-scaled VSV.
5. Organ and lesion mean doses with all engines; comparisons.
6. Maps and profiles.
7. Saving.

**About this notebook.** GATE Monte Carlo of the TIA maps in the patient CT and in water, against VSV.
- **Inputs:** nb04 outputs; `scripts/gate/lu177_patient_mc.py`.
- **Outputs:** `data/processed/deepblue/<patient>/mc/`, `kinetics/organ_doses_mc.csv`.
- **Run time:** minutes with the cached runs; 6 runs of 5-9 min each from scratch.
- **Environment:** .venv (opengate).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import opengate
import pandas as pd
import SimpleITK as sitk

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
PROC = WS / "data" / "processed" / "deepblue"
KIN = PROC / "kinetics"
MC_SCRIPT = WS / "scripts" / "gate" / "lu177_patient_mc.py"
GATE_DATA = Path(opengate.__file__).parent / "data"
PATIENTS = ["patient_4", "patient_6"]
SNMMI_LABEL = {"patient_4": "A", "patient_6": "B"}

N_DECAYS = 2_000_000
THREADS = 8
RUNS = [("hybrid", "ct"), ("hybrid", "water"), ("MIM", "ct")]

## 1. Inputs

All simulations use the 4 mm grid of notebook 04, on which the TIA maps already exist (`dose/tia_<map>_4mm.nii.gz`), so that source, phantom and scored dose share the same voxels:
- **CT**: the scan-1 CT, resampled from the SPECT grid to the 4 mm grid with linear interpolation (air outside the field: −1000 HU). Hounsfield units are converted to materials and densities with the Schneider et al. (2000) tables distributed with GATE (density tolerance 0.05 g/cm³). The CTs of this dataset are low-dose, non-contrast acquisitions, so no contrast correction is needed.
- **Water phantom**: the same image volume with every voxel set to water, including the air around the patient. This is the geometry implied by the VSV kernel, which assumes infinite water.
- **Source**: the TIA map in Bq·s/mL, with negative values set to zero, used as the probability map of the decay positions (uniform within each voxel).

In [ ]:
def read(path):
    return sitk.ReadImage(str(path))


def arr(image):
    return sitk.GetArrayFromImage(image).astype(np.float64)


def like(array, reference):
    image = sitk.GetImageFromArray(array)
    image.CopyInformation(reference)
    return image


hu_table = np.loadtxt(GATE_DATA / "Schneider2000DensitiesTable.txt", comments="#")


def hu_to_density(hu):
    return np.interp(hu, hu_table[:, 0], hu_table[:, 1])


data = {}
for patient in PATIENTS:
    pdir = PROC / patient
    mdir = pdir / "mc"
    (mdir / "inputs").mkdir(parents=True, exist_ok=True)
    grid = read(pdir / "dose" / "tia_hybrid_4mm.nii.gz")
    ct = sitk.Resample(read(pdir / "scan1" / "ct_on_spect.nii.gz"), grid, sitk.Transform(), sitk.sitkLinear, -1000.0,
                       sitk.sitkFloat32)
    sitk.WriteImage(sitk.Cast(ct, sitk.sitkInt16), str(mdir / "inputs" / "ct_4mm.mhd"))
    tia = {}
    for name in ["hybrid", "MIM"]:
        img = sitk.Cast(sitk.Clamp(read(pdir / "dose" / f"tia_{name}_4mm.nii.gz"), lowerBound=0.0), sitk.sitkFloat32)
        sitk.WriteImage(img, str(mdir / "inputs" / f"source_{name}_4mm.mhd"))
        tia[name] = arr(img)
    data[patient] = {"grid": grid, "mdir": mdir, "ct_hu": arr(ct), "tia": tia,
                     "ref": read(pdir / "scan1" / "spect_bqml.nii.gz"),
                     "masks": {p.name.removesuffix(".nii.gz").split("_", 1)[1]: arr(read(p)) > 0
                               for p in sorted((pdir / "scan1" / "masks").glob("*.nii.gz"))}}
    rho = hu_to_density(data[patient]["ct_hu"])
    body = data[patient]["ct_hu"] > -300
    print(patient, "grid", grid.GetSize(), f"median body density {np.median(rho[body]):.3f} g/cm3")

## 2. Monte Carlo runs

Each configuration is simulated by the script `lu177_patient_mc.py` in a separate process (Geant4 can be initialised only once per process). Settings:
- primary: Lu-177 ion at rest, full radioactive decay (Geant4 radioactive decay, electrons, photons and X-rays transported); $N = 2 \times 10^6$ decays per configuration, 8 threads;
- physics list `G4EmStandardPhysics_option4`; production cut 1 mm in the patient, 1 m in the surrounding air;
- dose scored on the 4 mm grid (`DoseActor`, dose to the local medium), with the per-voxel statistical uncertainty.

With these settings a pilot run of $2 \times 10^5$ decays gave a statistical uncertainty of about 1% on organ means and 2% on a 4 mL lesion; $2 \times 10^6$ decays bring them to about 0.3% and 0.7%. A configuration takes about 10 minutes on 8 threads. Runs that already exist on disk with the same number of decays are not repeated.

In [ ]:
runs = []
for patient in PATIENTS:
    mdir = data[patient]["mdir"]
    for name, phantom in RUNS:
        out = mdir / f"{name}_{phantom}"
        summary = out / "run.json"
        if not (summary.exists() and json.loads(summary.read_text())["n_requested"] == N_DECAYS):
            cmd = [sys.executable, str(MC_SCRIPT), "--ct", str(mdir / "inputs" / "ct_4mm.mhd"),
                   "--source", str(mdir / "inputs" / f"source_{name}_4mm.mhd"), "--output", str(out),
                   "--phantom", phantom, "--n", str(N_DECAYS), "--threads", str(THREADS)]
            subprocess.run(cmd, check=True, capture_output=True)
        info = json.loads(summary.read_text())
        runs.append({"patient": patient, "map": name, "phantom": phantom, "events": info["events"],
                     "minutes": info["seconds"] / 60})
pd.DataFrame(runs).set_index(["patient", "map", "phantom"]).round(1)

## 3. Scaling and statistical uncertainty

GATE scores the absorbed dose (Gy) deposited by the $N$ simulated decays. The source is the TIA map, so the total number of decays in the patient is $\tilde A_\text{tot} = \sum_\mathbf{x} \text{TIA}(\mathbf{x})\, V_\text{voxel}$ (Bq·s), and the absorbed dose is

$$D(\mathbf{x}) = D_\text{MC}(\mathbf{x})\, \frac{\tilde A_\text{tot}}{N}.$$

The statistical uncertainty of an organ mean is computed from the per-voxel relative uncertainty $u_i$ as $\sigma_{\bar D} = \sqrt{\sum_i (u_i D_i)^2}/n$, which neglects the correlation between neighbouring voxels.

In [ ]:
for patient in PATIENTS:
    d = data[patient]
    v_ml = float(np.prod(d["grid"].GetSpacing())) / 1000.0
    d["mc"], d["mc_unc"] = {}, {}
    for name, phantom in RUNS:
        out = d["mdir"] / f"{name}_{phantom}"
        dose = read(out / "mc_dose.mhd")
        assert np.allclose(dose.GetOrigin(), d["grid"].GetOrigin()) and dose.GetSize() == d["grid"].GetSize()
        scale = d["tia"][name].sum() * v_ml / N_DECAYS
        d["mc"][(name, phantom)] = arr(dose) * scale
        d["mc_unc"][(name, phantom)] = arr(read(out / "mc_dose_uncertainty.mhd"))

## 4. Density-scaled VSV

The VSV dose of notebook 04 assumes water. The simplest correction for the patient density is to divide the dose by the local density, $D_{\text{VSV}/\rho}(\mathbf{x}) = D_\text{VSV}(\mathbf{x}) / \rho(\mathbf{x})$, with $\rho$ in g/cm³ from the CT through the same Schneider density curve used by GATE. The correction is applied where $\rho > 0.1$ g/cm³; it is exact only for a dose that is deposited locally in a homogeneous medium, which is close to the situation inside soft-tissue organs.

In [ ]:
for patient in PATIENTS:
    d = data[patient]
    rho = hu_to_density(d["ct_hu"])
    d["vsv"] = {}
    for name in ["hybrid", "MIM"]:
        vsv = arr(read(PROC / patient / "dose" / f"dose_{name}_VSV_4mm.nii.gz"))
        d["vsv"][(name, "water")] = vsv
        d["vsv"][(name, "rho")] = np.where(rho > 0.1, vsv / np.maximum(rho, 0.1), vsv)

## 5. Organ and lesion doses

As in notebook 04, every dose map on the 4 mm grid is resampled to the native scan-1 grid (linear interpolation) and averaged over the scan-1 masks: healthy liver (lesions excluded), kidneys separately and pooled, spleen, lesions.

Columns: `MC_CT` (reference), `MC_water`, `VSV` (water kernel), `VSV_rho` (density-scaled), with the statistical uncertainty of the Monte Carlo means in percent.

In [ ]:
def regions_of(masks):
    lesions = [n for n in masks if n.startswith("lesion")]
    any_lesion = np.any([masks[n] for n in lesions], axis=0)
    regions = {n: masks[n] & ~any_lesion for n in masks if n not in lesions + ["liver"]}
    regions["liver_healthy"] = masks["liver"] & ~any_lesion
    if {"ltkidney", "rtkidney"} <= set(regions):
        regions["kidneys"] = regions["ltkidney"] | regions["rtkidney"]
    regions.update({n: masks[n] for n in lesions})
    return regions


def to_native(dose, d):
    return arr(sitk.Resample(like(dose.astype(np.float32), d["grid"]), d["ref"], sitk.Transform(), sitk.sitkLinear, 0.0))


rows = []
for patient in PATIENTS:
    d = data[patient]
    regions = regions_of(d["masks"])
    for name in ["hybrid", "MIM"]:
        maps = {"VSV": d["vsv"][(name, "water")], "VSV_rho": d["vsv"][(name, "rho")]}
        for phantom in ["ct", "water"]:
            if (name, phantom) in d["mc"]:
                maps[f"MC_{phantom.upper() if phantom == 'ct' else phantom}"] = d["mc"][(name, phantom)]
        native = {k: to_native(v, d) for k, v in maps.items()}
        unc = {k: to_native(d["mc"][(name, ph)] * d["mc_unc"][(name, ph)], d)
               for k, ph in [("MC_CT", "ct"), ("MC_water", "water")] if (name, ph) in d["mc"]}
        for n, m in regions.items():
            row = {"patient": patient, "map": name, "structure": n}
            for k, v in native.items():
                row[k] = v[m].mean()
            for k, u in unc.items():
                row[f"{k}_stat_%"] = 100 * np.sqrt((u[m] ** 2).sum()) / m.sum() / row[k]
            rows.append(row)
mc_doses = pd.DataFrame(rows).set_index(["patient", "map", "structure"])
mc_doses["VSV_over_MC_CT"] = mc_doses.VSV / mc_doses.MC_CT
mc_doses["VSV_rho_over_MC_CT"] = mc_doses.VSV_rho / mc_doses.MC_CT
mc_doses["MC_water_over_VSV"] = mc_doses.MC_water / mc_doses.VSV
mc_doses["MC_CT_over_MC_water"] = mc_doses.MC_CT / mc_doses.MC_water
mc_doses.to_csv(KIN / "organ_doses_mc.csv")
mc_doses.xs("hybrid", level="map")[["MC_CT", "MC_CT_stat_%", "MC_water", "VSV", "VSV_rho", "MC_water_over_VSV",
                                    "MC_CT_over_MC_water", "VSV_over_MC_CT", "VSV_rho_over_MC_CT"]].round(3)

**Engine and density effects, MIM map.** The same comparison for the MIM map (no water run): if the engine effects are properties of the anatomy and not of the TIA map, the ratios should match those of the hybrid map.

In [ ]:
mc_doses.xs("MIM", level="map")[["MC_CT", "MC_CT_stat_%", "VSV", "VSV_rho", "VSV_over_MC_CT", "VSV_rho_over_MC_CT"]].round(3)

**TIA map vs dose engine.** Hybrid dose divided by MIM dose, with the VSV kernel (notebook 04) and with the Monte Carlo in the CT. If the two columns agree, the differences between the TIA maps found in notebook 04 do not depend on the dose engine.

In [ ]:
h, m_ = mc_doses.xs("hybrid", level="map"), mc_doses.xs("MIM", level="map")
map_effect = pd.DataFrame({"hybrid_over_MIM_VSV": h.VSV / m_.VSV, "hybrid_over_MIM_MC_CT": h.MC_CT / m_.MC_CT})
map_effect["difference"] = map_effect.hybrid_over_MIM_MC_CT - map_effect.hybrid_over_MIM_VSV
map_effect.round(3)

## 6. Maps and profiles

Coronal slice through the organs for the hybrid map: Monte Carlo dose in the CT, VSV dose, and the ratio VSV/MC where the Monte Carlo dose is above 5% of its 99th percentile. The last panel is a profile along the left–right axis through the centroid of the left kidney.

In [ ]:
for patient in PATIENTS:
    d = data[patient]
    mc, vsv, rho = d["mc"][("hybrid", "ct")], d["vsv"][("hybrid", "water")], hu_to_density(d["ct_hu"])
    kid = sitk.Resample(like(d["masks"]["ltkidney"].astype(np.float32), d["ref"]), d["grid"], sitk.Transform(),
                        sitk.sitkLinear, 0.0)
    z, y, x = np.round(np.argwhere(arr(kid) > 0.5).mean(axis=0)).astype(int)
    vmax = np.percentile(mc, 99.9)
    fig, axes = plt.subplots(1, 4, figsize=(19, 4.8))
    axes[0].imshow(mc[::-1, y, :], cmap="inferno", vmin=0, vmax=vmax, aspect="auto")
    axes[0].set_title("MC in CT, hybrid TIA [Gy]")
    axes[1].imshow(vsv[::-1, y, :], cmap="inferno", vmin=0, vmax=vmax, aspect="auto")
    axes[1].set_title("VSV (water), hybrid TIA [Gy]")
    ratio = np.full(mc.shape, np.nan)
    hot = mc > 0.05 * np.percentile(mc, 99)
    ratio[hot] = vsv[hot] / mc[hot]
    im = axes[2].imshow(ratio[::-1, y, :], cmap="RdBu_r", vmin=0.8, vmax=1.2, aspect="auto")
    axes[2].set_title("VSV / MC")
    fig.colorbar(im, ax=axes[2], fraction=0.046)
    for ax in axes[:3]:
        ax.axhline(mc.shape[0] - 1 - z, color="cyan", lw=0.5)
        ax.axis("off")
    xs = (np.arange(mc.shape[2]) - x) * 4.0
    for k, (img, label) in enumerate([(mc, "MC CT"), (d["mc"][("hybrid", "water")], "MC water"), (vsv, "VSV"),
                                      (d["vsv"][("hybrid", "rho")], "VSV / rho")]):
        axes[3].plot(xs, img[z, y, :], label=label, lw=1.2 if k else 2)
    ax2 = axes[3].twinx()
    ax2.plot(xs, rho[z, y, :], color="gray", lw=0.6, ls=":")
    ax2.set_ylabel("density [g/cm3]", color="gray")
    axes[3].set_xlabel("x from left-kidney centroid [mm]")
    axes[3].set_ylabel("Dose [Gy]")
    axes[3].legend(fontsize=8)
    axes[3].set_title("profile through the left kidney")
    fig.suptitle(f"{patient} (SNMMI patient {SNMMI_LABEL[patient]})")
    fig.tight_layout()

### Findings

- **Statistics.** With $2 \times 10^6$ decays per configuration the statistical uncertainty of the Monte Carlo means is below 0.5% for every structure (0.05–0.16% for organs, 0.06–0.47% for lesions), negligible against all the effects below. The single voxels are much noisier (median relative uncertainty 15–30% in the organs, 6–18% in the lesions), so the Monte Carlo maps are a reference for organ and lesion means, not for voxel-by-voxel comparisons; a voxel-level reference at a few percent would need about 25–50 times more decays (several hours per configuration).
- **Convolution vs transport in water (MC water / VSV): 0.99–1.01 for kidneys, spleen and lesions, 1.02 for the healthy liver.** The VSV convolution reproduces the Monte Carlo in water. The 2% excess in the liver is the energy deposited farther than 40 mm from the decay, which the truncated kernel does not contain: mostly photons from the rest of the body, whose share of the dose is largest in a low-dose organ next to a 116 mL lesion (patient 4) or to the kidneys.
- **Patient materials vs water (MC CT / MC water): 0.94–0.975.** The organs and lesions have mean densities of 1.02–1.07 g/cm³ (Schneider curve), and the dose to the local medium scales approximately as $1/\rho$: the effect is largest in the liver and in dense lesions (0.945–0.95) and smallest in the kidneys (0.96–0.975, density 1.02–1.03).
- **VSV in water vs MC in CT: the VSV overestimates the patient dose by 3–9%** (organs 3–5%, lesions 4–9%), the sum of the two effects above.
- **Density-scaled VSV vs MC in CT: 0.997–1.017 for kidneys, spleen and lesions (1.03 for lesion 1 of patient 6 with the MIM map), 0.97 for the healthy liver.** Dividing the VSV dose by the local density removes almost all the difference in the organs and lesions; in the liver it over-corrects by 2.5–3.3%, because the long-range photon contribution missing from the kernel (previous point) is not restored by the density scaling. For the dose engine of the project, VSV/ρ is within about 2% of the Monte Carlo for kidneys, spleen and lesions and within about 3% for the healthy liver.
- **The TIA-map differences do not depend on the dose engine.** The hybrid/MIM dose ratios obtained with the VSV and with the Monte Carlo in the CT differ by at most 0.016 for patient 4 and 0.023 for patient 6 (lesion 1, a 13 mL lesion), and by less than 0.007 for all organs. The conclusions of notebook 04 on the TIA maps (lesion 2 19–32% above MIM, the bias of the voxel fits in the noisy liver of patient 4) are therefore properties of the kinetics and of the alignment, not of the dose calculation.
- **Order of magnitude of the effects**, for organs: Monte Carlo statistics < 0.5% < engine effect after density scaling (≤ 3%) < density effect (3–6%) < TIA-map effect (up to 4% for organs with the hybrid map, 17% for the voxel map in the noisy liver, 20–30% for lesion 2). For a 4 mm grid and soft-tissue abdominal organs, the uncertainty of the dose is dominated by the TIA, consistent with the core message of the project.

## 7. Saving

In `data/processed/deepblue/<patient>/mc/`: the inputs (`inputs/ct_4mm.mhd`, `inputs/source_<map>_4mm.mhd`), the raw GATE outputs of each configuration (`<map>_<phantom>/`, dose per simulated decays, uncertainty, material labels, `run.json`) and the scaled Monte Carlo dose maps `dose_<map>_MC_<phantom>_4mm.nii.gz` (Gy). In `data/processed/deepblue/kinetics/`: `organ_doses_mc.csv`.

In [ ]:
for patient in PATIENTS:
    d = data[patient]
    for (name, phantom), dose in d["mc"].items():
        sitk.WriteImage(like(dose.astype(np.float32), d["grid"]), str(d["mdir"] / f"dose_{name}_MC_{phantom}_4mm.nii.gz"))
    print("saved", d["mdir"])